# GraphCast: control vs. K&C bogus for Typhoon Mawar (WP02 2023)

Runs DeepMind's GraphCast (WeatherNext 1 Graph, 0.25°, 37 pressure levels,
public ERA5 weights) twice from the same ERA5 analysis at **2023-05-26 06Z**,
about a day before Mawar's peak:

1. **control**: the unaltered ERA5 state;
2. **bogus**: the ERA5 state with a Kwon & Cheong (2009) vortex matched to the
   JTWC best-track intensity.

GraphCast's native step is 6 h, so both runs go 72 h in 12 steps. At each step
the storm is located by `tcinit.rollout_tracking.track_rollout`, which centres
each box on the previous step's position, and the two runs are compared
against the best track.

**Run on:** a cee-compute GPU node in `jax-venv` (see `envs/README.md`). The
checkpoint, normalization statistics and ERA5 are read anonymously from
Google Cloud Storage, so the node needs outbound network access.

**Checkpoint.** `GraphCast - ERA5 1979-2017 - resolution 0.25 - pressure
levels 37 - mesh 2to6 - precipitation input and output`. It is the paper
model and expects ERA5 inputs, including 6-hourly accumulated precipitation.
Mawar (2023) is outside its training period. (`GraphCast_operational`, 13
levels, no precipitation input, is the HRES-initialised alternative.)

**Time ordering.** WeatherNext inputs carry two frames on a `time` axis of
lead times `[-6 h, 0 h]`, oldest first, so the current state is last
(`time_idx=-1`). Section 3 ends with a check of this against the best
track, before anything runs.

## 1. Configuration

In [ ]:
import os
from datetime import datetime, timedelta
from pathlib import Path

# Pick one A100. Must be set before jax is imported.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
# JAX grabs 75% of GPU memory on first use. Uncomment on a shared node.
# os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

PROJECT = Path("/projects/aurora")
BEST_TRACK_FILE = PROJECT / "best-tracks/westpac/bwp2023/bwp022023.dat"
ERA5_ZARR = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
WN_BUCKET = "dm_graphcast"
PARAMS_FILE = (
    "graphcast/params/GraphCast - ERA5 1979-2017 - resolution 0.25 - "
    "pressure levels 37 - mesh 2to6 - precipitation input and output.npz"
)
STATS_DIR = "graphcast/stats"
MODEL_CACHE = Path(f"/fgs/{os.environ['USER']}/causaltc/model-cache/weathernext")
OUT_DIR = Path(f"/fgs/{os.environ['USER']}/causaltc/demo_out/graphcast_mawar")

START_TIME = datetime(2023, 5, 26, 6)
STEP = timedelta(hours=6)  # GraphCast's native step
N_STEPS = 12  # 72 h
BOX_HALF_DEG = 10.0

if not BEST_TRACK_FILE.exists():
    raise FileNotFoundError(BEST_TRACK_FILE)
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"output -> {OUT_DIR}")

In [ ]:
import dataclasses
import functools

import gcsfs
import haiku as hk
import jax
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from weathernext.utils import (
    autoregressive,
    casting,
    checkpoint,
    data_utils,
    normalization,
    rollout,
)
from weathernext.weathernext1_graph import graphcast

from tcinit import BestTrack
from tcinit.backends import weathernext_graph_backend as wn_backend
from tcinit.constants import HPA_TO_PA
from tcinit.naming import WEATHERNEXT_VAR_MAP
from tcinit.plotting import plot_snapshot
from tcinit.rollout_snapshots import plot_rollout_comparison, plot_rollout_snapshots
from tcinit.rollout_tracking import build_bogus, snapshot_from_box, track_rollout

print("jax", jax.__version__, "on", jax.devices())

## 2. Best track

In [ ]:
bt = BestTrack(BEST_TRACK_FILE)
bt_center = bt.get_location(START_TIME)
print(
    f"{bt.ID} at {START_TIME:%Y-%m-%d %HZ}: "
    f"({bt_center[0]:.1f}N, {bt_center[1]:.1f}E), "
    f"{bt.get_windspeed(START_TIME):.0f} m/s, "
    f"{bt.get_central_pressure(START_TIME) / HPA_TO_PA:.0f} hPa"
)

## 3. Initial state from ERA5

The checkpoint is read first, because its `task_config` lists the input
variables and pressure levels; the network itself is built in section 6.
Files from the `dm_graphcast` bucket are cached under `MODEL_CACHE`.

GraphCast takes two input frames, `[t − 6 h, t]`. The fields come from the
ARCO ERA5 store, renamed to WeatherNext's `lat`/`lon` and flipped to
ascending latitude. `total_precipitation_6hr` is the sum of the six hourly
ERA5 accumulations ending at each input time.

The inputs, targets template and forcings are then built the way the
WeatherNext demo builds them: one dataset with the two input frames followed
by `N_STEPS` target frames, split by `data_utils.extract_inputs_targets_forcings`.
That function also computes the forcings (`toa_incident_solar_radiation` and
the year/day-progress features) at every frame from the `datetime` coordinate.
The target frames are lazy all-NaN dask arrays, so only one step of the
template is materialised at a time during the rollout.

In [ ]:
GCS = gcsfs.GCSFileSystem(token="anon")


def cached(path):
    # Local copy of gs://dm_graphcast/<path>, downloaded on first use.
    local = MODEL_CACHE / path
    if not local.exists():
        local.parent.mkdir(parents=True, exist_ok=True)
        GCS.get(f"{WN_BUCKET}/{path}", str(local))
    return local


with open(cached(PARAMS_FILE), "rb") as f:
    ckpt = checkpoint.load(f, graphcast.CheckPoint)
model_config, task_config = ckpt.model_config, ckpt.task_config
print(ckpt.description)
print(model_config)
print("levels:", task_config.pressure_levels, " input duration:", task_config.input_duration)

In [ ]:
STEP_H = int(STEP / timedelta(hours=1))
N_FRAMES = 2 + N_STEPS  # two input frames, then the target lead times
DATETIMES = [START_TIME + (k - 1) * STEP for k in range(N_FRAMES)]
HISTORY = tuple(DATETIMES[:2])  # oldest first, current last
LEVELS = list(task_config.pressure_levels)
STATIC = ("geopotential_at_surface", "land_sea_mask")
# Time-dependent inputs read from ERA5; the forcings are computed below.
DYNAMIC = [
    v
    for v in task_config.input_variables
    if v not in task_config.forcing_variables and v not in STATIC
]
TARGET_ONLY = [v for v in task_config.target_variables if v not in DYNAMIC]

era5 = xr.open_zarr(ERA5_ZARR, chunks=None, storage_options=dict(token="anon"))


def to_wn_grid(ds):
    # ARCO (latitude 90 -> -90, longitude) -> WeatherNext (lat -90 -> 90, lon).
    ds = ds.rename(latitude="lat", longitude="lon").sortby("lat")
    ds = ds.assign_coords(lat=ds.lat.astype(np.float32), lon=ds.lon.astype(np.float32))
    if "level" in ds.coords:
        ds = ds.assign_coords(level=ds.level.astype(np.int32))
    return ds


def era5_field(name, t):
    if name == "total_precipitation_6hr":
        hourly = era5["total_precipitation"].sel(
            time=slice(np.datetime64(t - timedelta(hours=5)), np.datetime64(t))
        )
        assert hourly.sizes["time"] == 6, hourly.time.values
        da = hourly.sum("time")
    else:
        da = era5[name].sel(time=np.datetime64(t))
    if "level" in da.dims:
        da = da.sel(level=LEVELS)
    return da.reset_coords(drop=True).astype(np.float32)


LEAD = (np.arange(N_FRAMES) * STEP_H).astype("timedelta64[h]").astype("timedelta64[ns]")
frames = xr.concat(
    [
        xr.Dataset({v: era5_field(v, t) for v in DYNAMIC}).load().expand_dims(time=[LEAD[k]])
        for k, t in enumerate(HISTORY)
    ],
    dim="time",
)
statics = xr.Dataset(
    {v: era5[v].sel(time=np.datetime64(START_TIME)).reset_coords(drop=True) for v in STATIC}
).load()
frames, statics = to_wn_grid(frames), to_wn_grid(statics)

# Lazy NaN frames for the target lead times.
pad = xr.full_like(frames.isel(time=[-1]).chunk(), np.nan)
pads = xr.concat([pad.assign_coords(time=[LEAD[k]]) for k in range(2, N_FRAMES)], dim="time")
dyn = xr.concat([frames, pads], dim="time")
for v in TARGET_ONLY:
    dyn[v] = xr.full_like(dyn["2m_temperature"], np.nan)
example = dyn.expand_dims("batch").assign({v: statics[v] for v in STATIC})
example = example.assign_coords(
    datetime=(("batch", "time"), np.array(DATETIMES, dtype="datetime64[ns]")[None])
)

inputs, targets_template, forcings = data_utils.extract_inputs_targets_forcings(
    example,
    target_lead_times=slice(f"{STEP_H}h", f"{N_STEPS * STEP_H}h"),
    **dataclasses.asdict(task_config),
)
inputs = inputs.compute()
forcings = forcings.compute()
assert model_config.resolution in (0, 360.0 / inputs.sizes["lon"])

print("inputs:   ", dict(inputs.sizes), inputs.time.values.astype("timedelta64[h]"))
print("targets:  ", dict(targets_template.sizes))
print("forcings: ", dict(forcings.sizes), list(forcings.data_vars))

In [ ]:
# Ordering check: slot i should hold the storm at HISTORY[i].
for i, t in enumerate(HISTORY):
    center = bt.get_location(t)
    box = wn_backend.extract_snapshot_dataset(
        inputs, box_center=center, box_size=BOX_HALF_DEG, time_idx=i
    )
    s = snapshot_from_box(
        box, valid_time=t, search_center=center, var_map=WEATHERNEXT_VAR_MAP
    )
    print(
        f"slot {i} ({t:%d %HZ}): model storm at ({s.loc[0]:.2f}, {s.loc[1]:.2f}), "
        f"{s.mslp_min / HPA_TO_PA:.1f} hPa | best track at ({center[0]:.2f}, {center[1]:.2f})"
    )

## 4. Control initial state

`time_idx=-1` is the current slot (lead time 0 h). This is also
`wn_backend.extract_snapshot_dataset`'s default; it is passed explicitly here
because the same `extract_box` reads the forecast chunks, which have a single
time slot. The box keeps WeatherNext's long-form variable names
(`mean_sea_level_pressure`, `temperature`, ...); `var_map=WEATHERNEXT_VAR_MAP`
tells `snapshot_from_box`, `build_bogus`, `track_rollout` and `plot_snapshot`
how to read them.

In [ ]:
def extract_box(state, center):
    return wn_backend.extract_snapshot_dataset(
        state, box_center=center, box_size=BOX_HALF_DEG, time_idx=-1
    )


box_ctrl0 = extract_box(inputs, bt_center)
snap_ctrl0 = snapshot_from_box(
    box_ctrl0, valid_time=START_TIME, search_center=bt_center, var_map=WEATHERNEXT_VAR_MAP
)
plot_snapshot(
    snap_ctrl0,
    box_ctrl0,
    var_map=WEATHERNEXT_VAR_MAP,
    title=f"Mawar, ERA5 {START_TIME:%Y-%m-%d %HZ} (control initial state)",
)
plt.show()

## 5. Bogus initial state

`build_bogus` detects the storm, builds the K&C vortex from the best-track
intensity, and applies it to the box with the backend's `apply`.
`wn_backend.write_back` returns a deep copy of `inputs` with the box pasted in.
It writes `msl`, `2t`, `10u`, `10v` and `t`, `u`, `v`, `q`, `z` on every level
the box shares with the inputs, across all batch entries and, by default
(`time_slots=None`), into **both** input frames. This matches the Aurora demo,
but it means that inside the box the bogus run's t − 6 h frame equals its
t = 0 frame. Pass `time_slots=(1,)` to modify only the current frame. Fields
the vortex does not model (`vertical_velocity`, `total_precipitation_6hr`,
the forcings and statics) are left unchanged.

In [ ]:
_, bv, box_bogus0 = build_bogus(
    box_ctrl0,
    bt,
    START_TIME,
    search_center=bt_center,
    var_map=WEATHERNEXT_VAR_MAP,
    apply=wn_backend.apply,
)
inputs_bogus = wn_backend.write_back(
    inputs,
    box_bogus0,
    box_lats=box_ctrl0.latitude.values,
    box_lons=box_ctrl0.longitude.values,
)

print(f"p_c  = {bv.p_c / HPA_TO_PA:.1f} hPa (best track)")
print(f"V_m  = {bv.V_m:.1f} m/s")
print(f"R_v  = {bv.R_v_km:.0f} km")

box_check = extract_box(inputs_bogus, bt_center)
snap_bogus0 = snapshot_from_box(
    box_check, valid_time=START_TIME, search_center=bt_center, var_map=WEATHERNEXT_VAR_MAP
)
print(
    f"MSLP min: control {snap_ctrl0.mslp_min / HPA_TO_PA:.1f} hPa -> "
    f"bogus {snap_bogus0.mslp_min / HPA_TO_PA:.1f} hPa"
)
plot_snapshot(
    snap_bogus0,
    box_check,
    var_map=WEATHERNEXT_VAR_MAP,
    title=f"Mawar, ERA5 + K&C bogus {START_TIME:%Y-%m-%d %HZ}",
)
plt.show()

## 6. Load GraphCast

The predictor is wrapped as in the WeatherNext GraphCast demo: bfloat16
casting, then input normalization and residual prediction, then the
autoregressive wrapper. `run_forward_jitted` predicts one 6 h step; the first
call compiles it, which takes a few minutes.

In [ ]:
diffs_stddev_by_level = xr.load_dataset(cached(f"{STATS_DIR}/diffs_stddev_by_level.nc"))
mean_by_level = xr.load_dataset(cached(f"{STATS_DIR}/mean_by_level.nc"))
stddev_by_level = xr.load_dataset(cached(f"{STATS_DIR}/stddev_by_level.nc"))
params, state = ckpt.params, {}


def construct_wrapped_graphcast(model_config, task_config):
    predictor = graphcast.GraphCast(model_config, task_config)
    predictor = casting.Bfloat16Cast(predictor)
    predictor = normalization.InputsAndResiduals(
        predictor,
        diffs_stddev_by_level=diffs_stddev_by_level,
        mean_by_level=mean_by_level,
        stddev_by_level=stddev_by_level,
    )
    return autoregressive.Predictor(predictor, gradient_checkpointing=True)


@hk.transform_with_state
def run_forward(model_config, task_config, inputs, targets_template, forcings):
    predictor = construct_wrapped_graphcast(model_config, task_config)
    return predictor(inputs, targets_template=targets_template, forcings=forcings)


def with_configs(fn):
    return functools.partial(fn, model_config=model_config, task_config=task_config)


def with_params(fn):
    return functools.partial(fn, params=params, state=state)


def drop_state(fn):
    return lambda **kw: fn(**kw)[0]


run_forward_jitted = drop_state(with_params(jax.jit(with_configs(run_forward.apply))))

## 7. Run and track both forecasts

`graphcast_states` yields the input dataset, then each forecast step from
`rollout.chunked_prediction_generator` (one 6 h step per chunk, moved to host
memory). Each chunk has one `time` slot, at lead time k × 6 h, so `extract_box`
with `time_idx=-1` reads the right step, and states[0] is the analysis at
`START_TIME`. `track_rollout` consumes them one at a time and keeps only the
storm box. Both runs use the same RNG key (GraphCast is deterministic anyway).

In [ ]:
def graphcast_states(inp):
    yield inp
    for chunk in rollout.chunked_prediction_generator(
        run_forward_jitted,
        rng=jax.random.PRNGKey(0),
        inputs=inp,
        targets_template=targets_template,
        forcings=forcings,
        num_steps_per_chunk=1,
    ):
        # CHECK ON HPC: chunk arrives numpy-backed (as in rollout.chunked_prediction).
        yield jax.device_get(chunk)


common = dict(
    cyclone_id=bt.ID,
    start_time=START_TIME,
    initial_center=bt_center,
    step=STEP,
    var_map=WEATHERNEXT_VAR_MAP,
)
ctrl = track_rollout(graphcast_states(inputs), extract_box, **common)
bogus = track_rollout(
    graphcast_states(inputs_bogus), extract_box, intervened=True, **common
)

ctrl.to_netcdf(OUT_DIR / "control")
bogus.to_netcdf(OUT_DIR / "bogus")

## 8. Compare with the best track

In [ ]:
cols = {
    "model_lead_hours": "lead_h",
    "model_MSLP": "MSLP",
    "model_VMAX": "VMAX",
    "position_error_km": "pos_err_km",
}
df_ctrl = ctrl.to_comparison_dataframe(bt)
df_bogus = bogus.to_comparison_dataframe(bt)
table = pd.concat(
    {
        "control": df_ctrl[list(cols)].rename(columns=cols),
        "bogus": df_bogus[list(cols)].rename(columns=cols).drop(columns="lead_h"),
        "best track": df_ctrl[["bt_MSLP", "bt_VMAX"]].rename(
            columns={"bt_MSLP": "MSLP", "bt_VMAX": "VMAX"}
        ),
    },
    axis=1,
)
table.round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for df, label, style in ((df_ctrl, "control", "o-"), (df_bogus, "bogus", "^--")):
    lead = df["model_lead_hours"]
    axes[0].plot(lead, df["model_MSLP"], style, label=label)
    axes[1].plot(lead, df["model_VMAX"], style, label=label)
    axes[2].plot(lead, df["position_error_km"], style, label=label)
axes[0].plot(df_ctrl["model_lead_hours"], df_ctrl["bt_MSLP"], "k-", lw=2, label="best track")
axes[1].plot(df_ctrl["model_lead_hours"], df_ctrl["bt_VMAX"], "k-", lw=2, label="best track")
for ax, ylabel in zip(axes, ("min MSLP (hPa)", "max 10 m wind (kt)", "position error (km)")):
    ax.set_xlabel("lead time (h)")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle(f"GraphCast, Mawar from {START_TIME:%Y-%m-%d %HZ}")
plt.tight_layout()
plt.show()

In [ ]:
plot_rollout_comparison(
    ctrl,
    bogus,
    OUT_DIR / "comparison_graphcast_mawar.pdf",
    best_track=bt,
    labels=("GraphCast control", "GraphCast + K&C bogus"),
)
plot_rollout_snapshots(ctrl, OUT_DIR / "rollout_graphcast_control.pdf", best_track=bt)
plot_rollout_snapshots(bogus, OUT_DIR / "rollout_graphcast_bogus.pdf", best_track=bt)
print(sorted(p.name for p in OUT_DIR.glob("*.pdf")))